# preflight Simple Example

A realistic story:
1. Load CSV data
2. Run preflight in `ci-balanced` mode
3. Identify blockers
4. Apply one remediation
5. Re-run and confirm gate improvement

In [1]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import Markdown, display
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

import preflight

In [2]:
# Build a small CSV from a public dataset as stand-in for user data.
ds = load_breast_cancer(as_frame=True)
raw_df = ds.frame[["mean radius", "mean texture", "mean smoothness", "target"]].copy()

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)
csv_path = data_dir / "simple_example_dataset.csv"
raw_df.to_csv(csv_path, index=False)

display(pd.DataFrame([{
    "csv_path": str(csv_path),
    "rows": len(raw_df),
    "columns": len(raw_df.columns),
    "target": "target",
}]))

,csv_path,rows,columns,target
0,data/simple_example_dataset.csv,569,4,target


In [3]:
target = "target"
loaded_df = pd.read_csv(csv_path)
train_df, valid_df = train_test_split(
    loaded_df,
    test_size=0.25,
    random_state=42,
    stratify=loaded_df[target],
)

# Intentionally inject one duplicate into TRAIN to simulate a common data issue.
train_df = pd.concat([train_df, train_df.iloc[[0] * 40]], ignore_index=True)

before_dataset = preflight.run(train_df, target=target, profile="ci-balanced")
before_split = preflight.run_split(train_df, valid_df, profile="ci-balanced")

display(Markdown(f"### Before remediation: dataset gate **{before_dataset.gate.status}**, score **{before_dataset.score:.1f}**"))
display(Markdown(f"### Before remediation: split gate **{before_split.gate.status}**, score **{before_split.score:.1f}**"))

before_payload = before_dataset.to_dict()
before_findings = pd.DataFrame(before_payload["findings"])
before_blockers = before_findings[before_findings["severity"].isin(["error", "critical"])][
    ["severity", "check_id", "title", "suggested_action"]
]

display(Markdown("#### Blocking findings (before)"))
display(before_blockers.reset_index(drop=True))

### Before remediation: dataset gate **FAIL**, score **94.0**

### Before remediation: split gate **PASS**, score **98.0**

#### Blocking findings (before)

,severity,check_id,title,suggested_action
0,error,duplicates.exact,40 exact duplicate row(s) detected (8.6%).,None


In [4]:
# Remediation: drop exact duplicates, then rerun checks.
train_fixed = train_df.drop_duplicates().copy()

after_dataset = preflight.run(train_fixed, target=target, profile="ci-balanced")
after_split = preflight.run_split(train_fixed, valid_df, profile="ci-balanced")

after_payload = after_dataset.to_dict()
after_findings = pd.DataFrame(after_payload["findings"])
after_blockers = after_findings[after_findings["severity"].isin(["error", "critical"])][
    ["severity", "check_id", "title", "suggested_action"]
]

before_ids = sorted(before_blockers["check_id"].tolist()) if not before_blockers.empty else []
after_ids = sorted(after_blockers["check_id"].tolist()) if not after_blockers.empty else []

comparison = pd.DataFrame([
    {
        "metric": "dataset_gate",
        "before": before_dataset.gate.status,
        "after": after_dataset.gate.status,
        "delta": "improved" if before_dataset.gate.status != after_dataset.gate.status else "no-change",
    },
    {
        "metric": "dataset_score",
        "before": round(before_dataset.score, 1),
        "after": round(after_dataset.score, 1),
        "delta": round(after_dataset.score - before_dataset.score, 1),
    },
    {
        "metric": "dataset_blocker_count",
        "before": len(before_ids),
        "after": len(after_ids),
        "delta": len(after_ids) - len(before_ids),
    },
    {
        "metric": "dataset_blockers",
        "before": ", ".join(before_ids) if before_ids else "none",
        "after": ", ".join(after_ids) if after_ids else "none",
        "delta": "resolved" if before_ids and not after_ids else "partial/no-change",
    },
    {
        "metric": "split_gate",
        "before": before_split.gate.status,
        "after": after_split.gate.status,
        "delta": "improved" if before_split.gate.status != after_split.gate.status else "no-change",
    },
])

display(Markdown(f"### After remediation: dataset gate **{after_dataset.gate.status}**, score **{after_dataset.score:.1f}**"))
display(Markdown(f"### After remediation: split gate **{after_split.gate.status}**, score **{after_split.score:.1f}**"))
display(comparison)

print(json.dumps({
    "before_gate": before_payload["gate"],
    "after_gate": after_payload["gate"],
    "before_summary": before_payload["summary"],
    "after_summary": after_payload["summary"],
}, indent=2))

### After remediation: dataset gate **PASS**, score **100.0**

### After remediation: split gate **PASS**, score **98.0**

,metric,before,after,delta
0,dataset_gate,FAIL,PASS,improved
1,dataset_score,94.0,100.0,6.0
2,dataset_blocker_count,1,0,-1
3,dataset_blockers,duplicates.exact,none,resolved
4,split_gate,PASS,PASS,no-change


{
  "before_gate": {
    "status": "FAIL",
    "reasons": [
      "1 error finding(s) exceeded policy fail threshold"
    ]
  },
  "after_gate": {
    "status": "PASS",
    "reasons": []
  },
  "before_summary": {
    "severity_counts": {
      "info": 7,
      "warn": 0,
      "error": 1,
      "critical": 0
    },
    "domain_counts": {
      "data_quality": 3,
      "target_risk": 1,
      "split_integrity": 0,
      "schema_contract": 1,
      "stat_anomaly": 2,
      "advisory": 1
    },
    "suppressed_findings": 0
  },
  "after_summary": {
    "severity_counts": {
      "info": 8,
      "warn": 0,
      "error": 0,
      "critical": 0
    },
    "domain_counts": {
      "data_quality": 3,
      "target_risk": 1,
      "split_integrity": 0,
      "schema_contract": 1,
      "stat_anomaly": 2,
      "advisory": 1
    },
    "suppressed_findings": 0
  }
}
